# Unit 9 -- iterative LQR

**4SC000 Optimal Control and Reinforcement Learning**

iLQR does not treat the problem as a generic nonlinear program at all. It
exploits the *sequential* structure directly, alternating

- a **backward pass**: linearize the dynamics and quadratize the cost around
  the current nominal trajectory, then run a Riccati-like recursion -- the
  direct nonlinear generalization of Unit 5's LQR backward recursion;
- a **forward pass**: roll the resulting affine feedback law forward, with a
  backtracking line search.

Two things come out of this that the direct methods do not give: very fast
local convergence, and a **feedback policy** rather than just an open-loop
trajectory.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

# the swing-up problem used throughout Unit 9
x0      = np.array([0.0, 0.0])      # hanging down, at rest
xtarget = np.array([np.pi, 0.0])    # inverted, at rest
T, h    = 6.0, 60                   # horizon and number of steps
tau     = T / h
t_grid  = np.arange(h + 1) * tau
print(f"h = {h} steps of tau = {tau:.3f} s over T = {T} s")

In [ ]:
import ilqr

u_il, xs_il, cost_il, cost_hist = ilqr.ilqr_solve(x0, xtarget, h, tau)
print(f"converged cost = {cost_il:.4f}")
print(f"theta(T) = {xs_il[-1,0]:.4f}   thetadot(T) = {xs_il[-1,1]:+.4f}")
print(f"cost over the first six iterations: "
      f"{np.array2string(np.array(cost_hist[:6]), precision=3)}")

In [ ]:
plt.figure(figsize=(6.4, 3.6))
plt.semilogy(cost_hist, "o-", lw=1.8, ms=4)
plt.xlabel("iteration"); plt.ylabel("cost")
plt.title("iLQR converges in a handful of iterations")
plt.tight_layout(); plt.show()

The cost drops by three orders of magnitude in the first four iterations and
is essentially converged by the fifth -- the near-quadratic convergence
typical of Newton-type methods, and far fewer iterations than the
Levenberg-Marquardt solves that shooting and collocation need. That is the
direct benefit of exploiting the dynamic-programming structure instead of
handing a dense nonlinear program to a general-purpose solver.

## The local optimum, and who else finds it

iLQR's $\theta(t)$ dips **backward** before swinging up. It is worth being
precise about what this does and does not show.

In [ ]:
import shooting, collocation

u_ss, xs_ss, eff_ss = shooting.single_shooting_solve(x0, xtarget, h, tau)
u_ms, xs_ms, _, eff_ms = shooting.multiple_shooting_solve(x0, xtarget, h, 6, tau)

plt.figure(figsize=(7.0, 3.8))
plt.plot(t_grid, xs_il[:, 0], lw=2.6, label="iLQR")
plt.plot(t_grid, xs_ss[:, 0], lw=1.6, ls="--", label="single shooting")
plt.plot(t_grid, xs_ms[:, 0], lw=1.6, ls=":", label="multiple shooting")
plt.axhline(np.pi, color="0.5", ls=":"); plt.axhline(0.0, color="0.85", lw=0.8)
plt.xlabel("t"); plt.ylabel(r"$\theta(t)$"); plt.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"  iLQR              min theta = {xs_il[:,0].min():+.3f}   cost = {cost_il:.4f}")
print(f"  single shooting   min theta = {xs_ss[:,0].min():+.3f}   cost = {eff_ss:.4f}")
print(f"  multiple shooting min theta = {xs_ms[:,0].min():+.3f}   cost = {eff_ms:.4f}")

Single shooting dips back too, and lands on a cost indistinguishable from
iLQR's. (On the trajectory-comparison figure in the slides the two curves lie
exactly on top of one another, which is why only one dipping curve is
visible.)

So this is **not** a property of iLQR. It is a property of where both methods
were started: `ilqr_solve` and `single_shooting_solve` both begin from
$u \equiv 0$, while multiple shooting and collocation additionally seed a
state trajectory running from $x(0)$ to $x(T)$.

Warm-start iLQR the same way and it finds the other optimum as well.

In [ ]:
u_cold, _, cost_cold, _ = ilqr.ilqr_solve(x0, xtarget, h, tau)
u_warm, xs_warm, cost_warm, _ = ilqr.ilqr_solve(x0, xtarget, h, tau, u0=u_ms)

print(f"  iLQR from u = 0        cost = {cost_cold:.4f}")
print(f"  iLQR warm started      cost = {cost_warm:.4f}   "
      f"({100*(cost_cold-cost_warm)/cost_cold:.1f}% better)")

plt.figure(figsize=(6.6, 3.6))
plt.plot(t_grid, xs_il[:, 0], lw=2, label="iLQR from $u=0$")
plt.plot(t_grid, xs_warm[:, 0], lw=2, ls="--", label="iLQR warm started")
plt.axhline(np.pi, color="0.5", ls=":"); plt.axhline(0.0, color="0.85", lw=0.8)
plt.xlabel("t"); plt.ylabel(r"$\theta(t)$"); plt.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Where this sits in the course

- **Back to Unit 5.** The backward pass *is* the Riccati recursion, applied
  to a linearization of the dynamics around a nominal trajectory rather than
  to a globally linear model.
- **Back to Unit 8.** Run the forward pass for one step only, re-linearize at
  the freshly measured state and repeat: that is receding-horizon iLQR, which
  is essentially nonlinear MPC. Unit 8 solved a single QP per step because
  its dynamics were linear; here the "QP" is the by-product of one backward
  pass.
- **Forward to Unit 10.** iLQR still needs to *know* $f$ and $g$ and to
  differentiate them. When the model is unknown and all you can do is
  interact with the system, the same Bellman recursion has to be learned from
  data instead -- which is reinforcement learning.

Continue with `A_9_EVCaseStudies.ipynb`, where the methods are put to work on
two problems that cannot be done by hand.